# Self-supervised objectives

This notebook checks the local InfoNCE, masking, and DINO-style teacher contracts. It uses no downloaded images or checkpoints.

In [ ]:
from pathlib import Path
import importlib.util
import sys
lesson_rel = Path('phases/04-computer-vision/17-self-supervised-vision')
candidates = []
for base in (Path.cwd(), *Path.cwd().parents):
    candidates.extend((base / lesson_rel / 'code/main.py', base / 'code/main.py'))
code_path = next(p.resolve() for p in candidates if p.is_file())
spec = importlib.util.spec_from_file_location('cv04_l17_nb', code_path)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
assert spec.loader is not None
spec.loader.exec_module(module)
print(code_path)

In [ ]:
if not module.TORCH_AVAILABLE:
    import numpy as np
    z1 = np.eye(4)
    loss = module.numpy_info_nce(z1, z1, tau=0.2)
    visible, masked = module.numpy_mask_indices(16, mask_ratio=0.25, seed=7)
    teacher = module.numpy_dino_teacher(np.zeros((4, 6)))
    assert np.isfinite(loss) and len(visible) + len(masked) == 16 and np.allclose(teacher.sum(axis=1), 1)
    print({'Build-It': 'NumPy objectives', 'info_nce': loss, 'visible': len(visible), 'masked': len(masked), 'Use-It': 'PyTorch skipped cleanly'})
else:
    import torch
    z1 = torch.eye(4)
    z2 = torch.eye(4)
    loss = module.info_nce(z1, z2, tau=0.2)
    visible, masked = module.random_mask_indices(16, mask_ratio=0.25, seed=7)
    assert torch.isfinite(loss) and len(visible) + len(masked) == 16
    print({'info_nce': float(loss), 'visible': len(visible), 'masked': len(masked)})

The objective compares paired rows; it is not a claim that this tiny fixture is a trained representation.